### Spatial feature Extraction 
Creating the dataset of the functional nuclear powerplants (412) with 1500 fake sites that serves at the negative instances, the dataset is thus divided into three categories categories; High-Potential Site (0.75), Unsuitable site(target = 0.0), Existing Site (target = 1.0) for the ML model thus total of about 2000 instances are used to train an ML model, the spatial features that are to be used for the final dataset are :
* ID (type : Int)
* Latitude (type : Continuous)
* longitude (type : Continuous)
* Region (type : Char)
* seismic_pga (type : Continuous)
* dist_to_water
* dist_to_fault (type : binary)
* population
* is_protected (type : Binary)
* dist_to_volcano
* border_distance 
* target (type : Binary)

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
import reverse_geocoder as rg
import pycountry_convert as pc
import rasterio
from shapely.geometry import Point

In [18]:
# Cleaning the functional powerplant dataset 
df = pd.read_csv("/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/operating_plants.csv")
print(df.columns)
# Dropping unnecessary columns
df = df.drop(columns=['Id', 'Name', 'CountryCode',
       'Status', 'ReactorType', 'ReactorModel', 'ConstructionStartAt',
       'OperationalFrom', 'OperationalTo', 'Capacity', 'LastUpdatedAt',
       'Source', 'IAEAId'])

# creating an Id column
df.insert(0, "Id", range(len(df)))


Index(['Id', 'Name', 'Latitude', 'Longitude', 'Country', 'CountryCode',
       'Status', 'ReactorType', 'ReactorModel', 'ConstructionStartAt',
       'OperationalFrom', 'OperationalTo', 'Capacity', 'LastUpdatedAt',
       'Source', 'IAEAId'],
      dtype='object')


In [ ]:
# creaing a region column
regions = {
    "AF": "Africa",
    "AS": "Asia",
    "EU": "Europe",
    "NA": "North America",
    "SA": "South America",
    "OC": "Oceania",
    "AN": "Antarctica"
}
def latlon_to_continent(lat, lon):
    try:
        result = rg.search((lat, lon))[0]
        country_code = result["cc"]
        cont_code = pc.country_alpha2_to_continent_code(country_code)
        print(f"Latitude: {lat}, Longitude: {lon} -> Country Code: {country_code}, Continent Code: {cont_code}")
        return regions[cont_code]
    except:
        return None

df["Region"] = df.apply(
    lambda row: latlon_to_continent(row["Latitude"], row["Longitude"]),
    axis=1
)

Loading formatted geocoded file...
Latitude: 69.709579, Longitude: 170.30625 -> Country Code: RU, Continent Code: EU
Latitude: 69.709579, Longitude: 170.30625 -> Country Code: RU, Continent Code: EU
Latitude: 39.807, Longitude: -5.698 -> Country Code: ES, Continent Code: EU
Latitude: 39.807, Longitude: -5.698 -> Country Code: ES, Continent Code: EU
Latitude: -23.008, Longitude: -44.457 -> Country Code: BR, Continent Code: SA
Latitude: -23.008, Longitude: -44.457 -> Country Code: BR, Continent Code: SA
Latitude: 35.31, Longitude: -93.23 -> Country Code: US, Continent Code: NA
Latitude: 35.31, Longitude: -93.229 -> Country Code: US, Continent Code: NA
Latitude: 40.182, Longitude: 44.147 -> Country Code: AM, Continent Code: AS
Latitude: 41.202, Longitude: 0.571 -> Country Code: ES, Continent Code: EU


Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=78, pipe_handle=97)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/miniconda3/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/opt/miniconda3/lib/python3.13/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
  File "/opt/miniconda3/lib/python3.13/site-packages/reverse_geocoder/__init__.py", line 19, in <module>
    from scipy.spatial import cKDTree as KDTree
  File "/opt/miniconda3/lib/python3.13/site-packages/scipy/spatial/__init__.py", line 123, in <module>
    from . import distance, transform
  File "/opt/miniconda3/lib/python3.13/site-packages/scipy/spatial/transform/__init__.py", line 19, in <module>
    from ._rotation import Rotation, Slerp
  File "_rotation.pyx", line 7

Latitude: -33.967, Longitude: -59.209 -> Country Code: AR, Continent Code: SA
Latitude: 52.092, Longitude: 47.952 -> Country Code: RU, Continent Code: EU
Latitude: 52.092, Longitude: 47.952 -> Country Code: RU, Continent Code: EU
Latitude: 52.092, Longitude: 47.952 -> Country Code: RU, Continent Code: EU
Latitude: 52.092, Longitude: 47.952 -> Country Code: RU, Continent Code: EU
Latitude: 23.952748, Longitude: 52.193298 -> Country Code: AE, Continent Code: AS
Latitude: 23.952748, Longitude: 52.193298 -> Country Code: AE, Continent Code: AS
Latitude: 23.952748, Longitude: 52.203298 -> Country Code: AE, Continent Code: AS
Latitude: 40.624, Longitude: -80.432 -> Country Code: US, Continent Code: NA
Latitude: 40.624, Longitude: -80.432 -> Country Code: US, Continent Code: NA


In [ ]:
# Creating target column
df["Target"] = 1.0

# creating seismic_pga, dist_to_water, dist_to_fault, population, is_protected_area, dist_to_volcano, border_cooperation_score
# using rasterio to read the raster files and extract the values for each plant location
RASTER_DIR   = "/Users/mohammadbilal/Documents/Projects/N-Project/data/processed/rasters"
OUTPUT_CSV = "/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/ml_dataset.csv"
RASTERS = {
    "seismic_pga":      f"{RASTER_DIR}/seismic_data.tif",
    "dist_to_water":    f"{RASTER_DIR}/distance_to_water.tif",
    "dist_to_fault":    f"{RASTER_DIR}/fault_exclusion_mask.tif",
    "population":       f"{RASTER_DIR}/population_data.tif",
    "is_protected":     f"{RASTER_DIR}/protected_areas_mask.tif",
    "dist_to_volcano":  f"{RASTER_DIR}/volcano_data.tif",
    "border_distance":  f"{RASTER_DIR}/distance_to_border.tif",
}

def sample_point(lon, lat, raster_handles):
    values = {}
    for name, src in raster_handles.items():
        try:
            row, col = src.index(lon, lat)
            window = rasterio.windows.Window(col, row, 1, 1)
            data = src.read(1, window=window)
            val = float(data[0, 0])
            nodata = src.nodata
            if nodata is not None:
                # For float max nodata, use tolerance; for everything else exact match
                if abs(nodata) > 1e307:
                    if abs(val - nodata) < 1e300:
                        val = np.nan
                else:
                    if val == nodata:
                        val = np.nan
        except Exception:
            val = np.nan
        values[name] = val
    return values

raster_handles = {name: rasterio.open(path) for name, path in RASTERS.items()}

# Sample rasters at every point
records = []
skipped = 0

for _, row in df.iterrows():
    vals = sample_point(row["Longitude"], row["Latitude"], raster_handles)
 
    records.append({
        "Id":        row["Id"],
        "Latitude":  row["Latitude"],
        "Longitude": row["Longitude"],
        "Country":   row["Country"],
        "Region":    row["Region"],
        "Target":    row["Target"],
        **vals,
    })

for src in raster_handles.values():
    src.close()

ml_df = pd.DataFrame(records)
ml_df.drop(columns=["Country"], inplace=True)
ml_df.to_csv(OUTPUT_CSV, index=False)

# Converting to GeoJSON
geometry = [
    Point(xy) for xy in zip(ml_df["Longitude"], ml_df["Latitude"])
]
gdf = gpd.GeoDataFrame(
    ml_df,
    geometry=geometry,
    crs="EPSG:4326"
)
gdf.to_file("/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/ml_dataset.geojson", driver="GeoJSON")

In [ ]:
# creeating negative sites dataset (with target=0.0)
RASTER_DIR = "/Users/mohammadbilal/Documents/Projects/N-Project/data/processed/rasters"
OUTPUT_CSV = "/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/negative_sites.csv"

BBOX = {"lon_min": -180, "lon_max": 180, "lat_min": -60, "lat_max": 85}
N = 1000


def is_bad_site(vals):
    seismic        = vals.get("seismic_pga",     np.nan)
    dist_water     = vals.get("dist_to_water",   np.nan)
    dist_fault     = vals.get("dist_to_fault",   np.nan)
    is_protected   = vals.get("is_protected",    np.nan)
    dist_volcano   = vals.get("dist_to_volcano", np.nan)
    border_dist    = vals.get("border_distance", np.nan)

    conditions = [
        (not np.isnan(is_protected)  and is_protected == 0),  # actually protected
        (not np.isnan(dist_volcano)  and dist_volcano < 50),  # near volcano
        (not np.isnan(dist_fault)    and dist_fault == 0),    # on fault
        (not np.isnan(dist_water)    and dist_water > 200),   # far from water
        (not np.isnan(seismic)       and seismic > 0.4),      # high seismic
        (not np.isnan(border_dist)   and border_dist < 20),   # too close to border
    ]
    return any(conditions)

def sample_point(lon, lat, raster_handles):
    values = {}
    for name, src in raster_handles.items():
        try:
            row, col = src.index(lon, lat)
            window = rasterio.windows.Window(col, row, 1, 1)
            data = src.read(1, window=window)
            val = float(data[0, 0])
            nodata = src.nodata
            if nodata is not None:
                if abs(nodata) > 1e307:
                    if abs(val - nodata) < 1e300:
                        val = np.nan
                else:
                    if val == nodata:
                        val = np.nan
        except Exception:
            val = np.nan
        values[name] = val
    return values


raster_handles = {name: rasterio.open(path) for name, path in RASTERS.items()}
records = []
attempts = 0
max_attempts = N * 50

print(f"Generating {N} negative points...")

while len(records) < N and attempts < max_attempts:
    attempts += 1
    lon = np.random.uniform(BBOX["lon_min"], BBOX["lon_max"])
    lat = np.random.uniform(BBOX["lat_min"], BBOX["lat_max"])

    vals = sample_point(lon, lat, raster_handles)

    # Skip if all NaN 
    if all(np.isnan(v) for v in vals.values()):
        continue

    # Only keep if it's a genuinely bad site
    if not is_bad_site(vals):
        continue

    records.append({
        "Id":        f"neg_{len(records)}",
        "Latitude":  lat,
        "Longitude": lon,
        "Country":   "Unknown",
        "Region":    "Unknown",
        "Target":    0.0,
        **vals,
    })

for src in raster_handles.values():
    src.close()

neg_df = pd.DataFrame(records)
neg_df.to_csv(OUTPUT_CSV, index=False)

# Converting to GeoJSON
geometry = [
    Point(xy) for xy in zip(neg_df["Longitude"], neg_df["Latitude"])
]
gdf = gpd.GeoDataFrame(
    neg_df,
    geometry=geometry,
    crs="EPSG:4326"
)
gdf.to_file("/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/negative_sites.geojson", driver="GeoJSON")

Generating 1000 negative points...


In [ ]:
# High-Potential Sites: target = 0.5 
HIGHPOT_OUTPUT_CSV = "/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/high_potential_sites.csv"
N_HIGHPOT = 500

def is_high_potential(vals):
    seismic        = vals.get("seismic_pga",     np.nan)
    dist_water     = vals.get("dist_to_water",   np.nan)
    dist_volcano   = vals.get("dist_to_volcano", np.nan)
    is_protected   = vals.get("is_protected",    np.nan)
    dist_fault     = vals.get("dist_to_fault",   np.nan)
    population     = vals.get("population",      np.nan)
    border_dist    = vals.get("border_distance", np.nan)

    conditions = [
        (not np.isnan(dist_water)   and dist_water <= 30),
        (not np.isnan(seismic)      and seismic < 0.065),
        (not np.isnan(dist_volcano) and dist_volcano > 300),
        (not np.isnan(is_protected) and is_protected == 1),   # safe from protected areas
        (not np.isnan(dist_fault)   and dist_fault == 1),     # safe from faults
        (not np.isnan(population)   and population < 1000),
        (not np.isnan(border_dist)  and border_dist > 50),    # comfortably inside border
    ]
    return all(conditions)

raster_handles = {name: rasterio.open(path) for name, path in RASTERS.items()}
highpot_records = []
attempts = 0
max_attempts = N_HIGHPOT * 50

print(f"Generating {N_HIGHPOT} high-potential sites...")

while len(highpot_records) < N_HIGHPOT and attempts < max_attempts:
    attempts += 1
    lon = np.random.uniform(BBOX["lon_min"], BBOX["lon_max"])
    lat = np.random.uniform(BBOX["lat_min"], BBOX["lat_max"])

    vals = sample_point(lon, lat, raster_handles)

    if all(np.isnan(v) for v in vals.values()):
        continue

    if not is_high_potential(vals):
        continue

    highpot_records.append({
        "Id":        f"highpot_{len(highpot_records)}",
        "Latitude":  lat,
        "Longitude": lon,
        "Country":   "Unknown",
        "Region":    "Unknown",
        "Target":    0.5,
        **vals,
    })
for src in raster_handles.values():
    src.close()

highpot_df = pd.DataFrame(highpot_records)
highpot_df.to_csv(HIGHPOT_OUTPUT_CSV, index=False)

if len(highpot_df) > 0:
    gdf_highpot = gpd.GeoDataFrame(
        highpot_df,
        geometry=[Point(xy) for xy in zip(highpot_df["Longitude"], highpot_df["Latitude"])],
        crs="EPSG:4326"
    )
    gdf_highpot.to_file(
        "/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/high_potential_sites.geojson",
        driver="GeoJSON"
    )

In [4]:
import pandas as pd

ML_CSV      = "/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/ml_dataset.csv"
NEG_CSV     = "/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/negative_sites.csv"
TRICKY_CSV  = "/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/tricky_negative_sites.csv"
HIGHPOT_CSV = "/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/high_potential_sites.csv"
OUTPUT_CSV  = "/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/final_dataset.csv"

ml_df      = pd.read_csv(ML_CSV)
neg_df     = pd.read_csv(NEG_CSV)
tricky_df  = pd.read_csv(TRICKY_CSV)
highpot_df = pd.read_csv(HIGHPOT_CSV)

final_df = pd.concat([ml_df, neg_df, tricky_df, highpot_df], ignore_index=True)
final_df.drop(columns=["Id"], inplace=True, errors="ignore")
final_df.drop(columns=["Country"], inplace=True, errors="ignore")
final_df.to_csv(OUTPUT_CSV, index=False)

# Converting to GeoJSON
geometry = [
    Point(xy) for xy in zip(final_df["Longitude"], final_df["Latitude"])
]
gdf = gpd.GeoDataFrame(
    final_df,
    geometry=geometry,
    crs="EPSG:4326"
)
gdf.to_file("/Users/mohammadbilal/Documents/Projects/N-Project/data/model_data/Dataset.geojson", driver="GeoJSON")
